# SIGNO data pipeline in Google Colab

**Purpose.** Runs the whole SIGNO pipeline (NASA Earthdata, Hansen/UMD and GRACE data for the Amazon) from download to the JSON, PNG and API files used by the web app.

**Inputs.** An Earthdata account (free), two boundary zips (RAISG Biomas 2025 and HydroBASINS), and the public repository `AetherKernel/SIGNO`.
**Outputs.** `web/public/data/` (JSON, `validation.json`, `maps/`) and `web/server/data/` (API files). Intermediate clips (`data/interim/`, about 1.9 GB) stay on the Colab disk.

**Time and volume.**
- About 1 h 22 min for the whole run with section 4b (measured in Colab on 15 Nov 2026: downloads 1 h 14 min, four at a time; exports, trend test and check 8 min).
- About 60 GB are transferred for the monthly MODIS products. Only one global file is on disk at a time.
- The final `export_json` + `export_maps` + `validate` run takes about 15 minutes.
- Free Colab sessions last at most 12 hours and reset when idle. Keep the tab open: if the session stops, the Colab disk is wiped and the run starts again from section 1. Within a session, re-running a cell skips the periods already finished.
- Each pipeline cell starts with `%%time`, so its output ends with the time it took.

AI-generated: this notebook was written with Claude (Anthropic) via Claude Code, 2026-11-15, following the team's pipeline; reviewed by team AETHER.

[Open in Colab](https://colab.research.google.com/github/AetherKernel/SIGNO/blob/main/notebooks/signo_pipeline_colab.ipynb)

## 1. Get the code and install the requirements
Clones the public repository once per session (skipped if the folder exists) and installs `pipeline/requirements.txt`. Takes 1-2 minutes.

If `pip` fails on `pyhdf`, install the HDF4 system library first with `!apt-get install -y libhdf4-dev` and run the cell again. It is not needed on Colab's usual wheels, so it is not run by default.

While installing, `pip` may print red lines such as "ERROR: pip's dependency resolver does not currently take into account all the packages that are installed" that list Colab's own packages (google-colab, tensorflow, gradio, numba, datasets, gcsfs). This is expected: the pipeline does not use those packages, every step runs in a new `python` process with the pinned versions, and there is no need to restart the session (if you do, run sections 1 and 2 again).

In [ ]:
import os

# Clone only if the folder is not there yet (a second clone would fail)
if not os.path.exists("/content/SIGNO"):
    !git clone https://github.com/AetherKernel/SIGNO.git /content/SIGNO
%cd /content/SIGNO
!pip install -q -r pipeline/requirements.txt

## 2. Earthdata credentials
Store your credentials as Colab **Secrets** (key icon in the left bar) and switch on **Notebook access** for each one:
- `EARTHDATA_TOKEN` (a user token from your Earthdata profile), **or**
- `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD`.

The cell tries the token first. Credentials are copied to environment variables and never printed. The `python -m` commands below inherit them.

Also check in your Earthdata profile (*Applications > Authorized Apps*) that **NASA GESDISC DATA ARCHIVE** is approved, or the GLDAS, IMERG and GRACE downloads will fail. Never write credentials in a cell: this notebook is public.

In [ ]:
import os
from google.colab import userdata

def secret(name):
    # Returns the secret, or None if it does not exist or notebook access is off
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret("EARTHDATA_TOKEN")
if token:
    os.environ["EARTHDATA_TOKEN"] = token
else:
    user, pwd = secret("EARTHDATA_USERNAME"), secret("EARTHDATA_PASSWORD")
    if user and pwd:
        os.environ["EARTHDATA_USERNAME"] = user
        os.environ["EARTHDATA_PASSWORD"] = pwd

# The pipeline's own login: it stops with an error if it cannot authenticate
from pipeline.download import login
auth = login()
print("Authenticated with Earthdata:", bool(auth and auth.authenticated))

## 3. Keep the intermediate data on the Colab disk
The pipeline writes its intermediate files (`data/interim/`, about 1.9 GB in 1,629 files) to the Colab disk, which is faster than Google Drive and needs no Drive access. The disk is wiped when the session stops, so keep the tab open and download the results in sections 15b and 16 before closing. This cell creates the folder and checks the free space.

In [ ]:
import os, shutil

os.makedirs("data/interim", exist_ok=True)
free_gb = shutil.disk_usage("/content").free / 1e9
print("data/interim ->", os.path.realpath("data/interim"))
print(f"Free disk: {free_gb:.0f} GB")
if free_gb < 20:
    print("Warning: less than 20 GB free. The Hansen tiles and the global MODIS files need room while they are clipped.")

## 4. Boundary files
- **HydroBASINS level 3 (South America)** is optional: it is only used if `AMAZON_BOUNDARY` is `hydrobasins_basin`. The cell tries to download it, but the server may refuse (HTTP 403 on 15 Nov 2026); that is not an error and does not stop the notebook.
- **RAISG Biomas 2025** has terms of use and a registration, so the notebook cannot download it. Get it from <https://www.raisg.org/en/download/biomes/>, then upload it (Files panel, left bar) to `data/raw/boundaries/Biomas_2025.zip`, keeping that exact name. `AMAZON_BOUNDARY` in `pipeline/config.py` is `raisg_biome`, so it is required.
- Natural Earth countries is downloaded by the pipeline itself.

In [ ]:
import os, urllib.request

os.makedirs("data/raw/boundaries", exist_ok=True)

hybas = "data/raw/boundaries/hybas_sa_lev03_v1c.zip"
if not os.path.exists(hybas):
    try:
        urllib.request.urlretrieve("https://data.hydrosheds.org/file/HydroBASINS/standard/hybas_sa_lev03_v1c.zip", hybas)
    except Exception as e:
        if os.path.exists(hybas):
            os.remove(hybas)
        print("HydroBASINS download failed (" + str(e) + "). It is OPTIONAL and not needed with the default RAISG boundary. To add it by hand, download https://data.hydrosheds.org/file/HydroBASINS/standard/hybas_sa_lev03_v1c.zip in a browser and upload it to data/raw/boundaries/.")
if os.path.exists(hybas):
    print("HydroBASINS:", os.path.getsize(hybas) // 1_000_000, "MB")

In [ ]:
import os

# Stop here with a clear message if the RAISG file has not been uploaded yet
if not os.path.exists("data/raw/boundaries/Biomas_2025.zip"):
    raise SystemExit("Missing data/raw/boundaries/Biomas_2025.zip. Download it from "
                     "https://www.raisg.org/en/download/biomes/ and upload it with the Files panel.")
print("RAISG Biomas 2025 found.")

## 4b. Run everything with one command (recommended)
`pipeline.run_all` checks the credentials, the RAISG zip and the disk first, then:

1. downloads the 8 sources **in parallel** (4 at a time, the longest first), one log per source in `data/interim/logs/`;
2. checks every log and stops if a month or tile failed (run this cell again: finished months are skipped);
3. runs the exports, the trend test and the country borders in order;
4. runs `pipeline.verify` against the published results and prints the time of every step.

Progress is printed every 2 minutes. **Keep this tab open** until it ends: the Colab disk is wiped when the session stops.

Time measured in Colab on 15 Nov 2026 (free tier, Python 3.13): **1 h 22 min in total**: downloads 1 h 14 min with 4 at a time (the slowest source, MCD64A1, sets the pace), then exports, trend test and check in 8 min. The check found the same results as the repository (147 files identical, 3 with only a reworded text, 0 different). Times depend on the NASA servers.

The dates are the ones of the published run (15 Nov 2026), so the check compares the same data. Add `--latest` to download up to the newest month NASA has (the check will then show longer monthly lists).

Results of this cell: `data/interim/logs/run_summary.json` (times) and `data/interim/logs/verify_report.json` (check).

In [ ]:
%%time
!python -m pipeline.run_all

## Optional: one step at a time (sections 5 to 14c)
Section 4b already runs all of these. Use them only to retry or inspect one step: set `STEP_BY_STEP = True` below, then run the cell of that step. With `False`, "Run all" skips them, so nothing runs twice.

In [ ]:
STEP_BY_STEP = False

## 5. Download and clip the monthly products
Four cells, one product each, so any of them can be re-run alone (with `STEP_BY_STEP = True`). Each month is downloaded, clipped to the Amazon box, checked, and the global file is deleted. Months already in `data/interim/` are skipped; failed months are listed at the end (run the cell again to retry).

Time: MOD13C2 and MOD11C3 are the slow ones (about 28 GB each, hours); GLDAS and IMERG are smaller.

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.download --products MOD13C2 --start 2000-01 --end 2026-10
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.download --products MOD11C3 --start 2000-01 --end 2026-10
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.download --products GLDAS_NOAH025_M --start 2000-01 --end 2026-07
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.download --products GPM_3IMERGM --start 2000-01 --end 2025-09
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

## 6. Tree cover tiles (MOD44B, yearly)
Reads the sinusoidal tiles over the Amazon box for each year and converts them to 0.05 degree cells. About 1.3 GB of tiles per year are on disk at a time and are deleted afterwards. Output: `data/interim/MOD44B/<YYYY>.nc`. Finished years are skipped.

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.tiles --product MOD44B --start 2000 --end 2025
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

## 7. Burned area tiles (MCD64A1, monthly)
Same idea per month (about 51 MB of tiles at a time). Output: `data/interim/MCD64A1/<YYYY-MM>.nc`. The newest month may not be published by NASA yet. Finished months are skipped.

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.tiles --product MCD64A1 --start 2000-11 --end 2026-09
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

## 8. Hansen tree cover 2000 and yearly loss
Downloads 16 Hansen/UMD tiles over HTTPS (no login, about 8.6 GB in total), aggregates each one to 0.05 degree cells and deletes its files. Output: `data/interim/HANSEN/gfc_v1.13.nc`. A finished tile is skipped on re-run.

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.hansen
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

## 9. GRACE water storage
Downloads one global GRACE/GRACE-FO file (about 44 MB) and clips it to the box on the 0.5 degree grid. Output: `data/interim/GRACE/tws.nc`. There is no retry loop: run the cell again if it fails.

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.grace
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

## 10. Export the JSON files
Builds the zone masks and the monthly, six-month and yearly values. Output: `web/public/data/amazon_monthly.json`, `amazon_semester.json` and `amazon_annual.json`. It prints the number of months found per source: **check that count** before publishing, because this step does not stop if a source is incomplete. About 10 minutes.

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.export_json
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

## 11. Export the map layers
Writes the 0.05 degree map layers as 8-bit PNG images and `maps/index.json` in `web/public/data/maps/`. Needs the MOD44B, MCD64A1 and Hansen outputs. A few minutes.

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.export_maps
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

## 12. Export the API files

Converts the results into the files the web app reads (`web/server/data/`: `region.json`, `variables.json`, `trends/`, `series/`), on 1° cells inside the Amazon. No trend test is applied (slopes and p-values stay null: the method is the team's decision). It only replaces those four items; `cover.json` and `validation.json` are written by the next steps.

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.export_api
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

## 13. Export tree cover per cell

Writes `web/server/data/cover.json`: yearly tree cover (Hansen-derived and MODIS MOD44B), tree cover loss and burned area per 1° cell. Needs the files from step 12. Takes a few minutes.

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.export_cover
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

## 14. Cross-validation

Compares MOD44B with Hansen, checks forest loss against burned area, lists hotspots and the La Pampa case. Writes `web/public/data/validation.json` and the web copy `web/server/data/validation.json`. Descriptive only: no significance test.

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.validate
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

## 14b. Trend test (step 8, run last)

Runs the team's trend test (ADR-006): Mann-Kendall on yearly values per 1° cell, Sen slope per decade, Benjamini-Hochberg FDR per variable, alpha 0.05, with the pre-registered rules. Writes the results into `web/server/data/trends/`, `cover.json`, `zones.json` and `validation.json` (`trendChecks`). It must run AFTER `export_api`, `export_cover` and `validate`: re-running any of those removes the trend results, so run this cell again afterwards.

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.test_trends   # unit tests of the trend code
    !python -m pipeline.trends
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

## 14c. Country borders for the map

Writes `web/public/data/borders.json` (Natural Earth, public domain), the outlines drawn over the map.

In [ ]:
%%time
if STEP_BY_STEP:
    !python -m pipeline.export_borders
else:
    print("Skipped: section 4b already ran this step (set STEP_BY_STEP = True to run it alone).")

## 15. Quick summary
Loads `validation.json` and `variables.json` and prints a few numbers as a sanity check.

In [ ]:
import json

with open("web/public/data/validation.json", encoding="utf-8") as f:
    validation = json.load(f)
print("validation.json top-level keys:", list(validation))
for key, value in validation.items():
    size = len(value) if hasattr(value, "__len__") else value
    print(f"  {key}: {type(value).__name__} ({size})")

with open("web/server/data/variables.json", encoding="utf-8") as f:
    variables = json.load(f)
print("\nvariables.json:", len(variables), "variables")
for v in variables[:10]:
    print("  ", v.get("id") if isinstance(v, dict) else v)

## 15b. Check that the results match the repository (reproducibility)
Section 4b already ran this check; run it again after any step-by-step run. Compares every file the pipeline just wrote (`web/server/data/`, `web/public/data/`, JSON and PNG) with the version committed in the repository (`git HEAD` of this clone). The run timestamps are ignored.

- **IDENTICAL**: same content. **ROUNDING**: numbers differ by less than 0.1 % (expected between Windows and Linux).
- **TEXT**: numbers and yes/no values match; only a text changed. Read the examples.
- **DIFFERENT**: a number beyond the tolerance, a cell whose significance flips, a list of another length or a missing file. Read the examples before trusting the run.

Expected, not an error: if NASA published a month after 15 Nov 2026 inside the requested range (for example burned area 2026-10), the monthly lists get one more item. The trend results use complete years only, so their numbers should still match.

The full report is saved as `data/interim/logs/verify_report.json`; section 16 downloads it.

In [ ]:
%%time
!python -m pipeline.verify --report data/interim/logs/verify_report.json

## 16. Get the results back into the repository
Zip the two output folders and download the zips to your computer. Then copy the files into your local clone and commit them through the team's normal workflow (branch, review, push).

**Never commit credentials**, the intermediate data or the boundary zips: only the generated files in `web/public/data/` and `web/server/data/`.

In [ ]:
import os, shutil
from google.colab import files

# One zip per output folder (make_archive takes a single base folder)
shutil.make_archive("/content/signo_public_data", "zip", root_dir="/content/SIGNO", base_dir="web/public/data")
shutil.make_archive("/content/signo_server_data", "zip", root_dir="/content/SIGNO", base_dir="web/server/data")
files.download("/content/signo_public_data.zip")
files.download("/content/signo_server_data.zip")

# The run times (section 4b) and the reproducibility report (sections 4b and 15b), when they exist
for f in ["data/interim/logs/run_summary.json", "data/interim/logs/verify_report.json"]:
    if os.path.exists(f):
        files.download(f)